In [1]:
!git clone https://github.com/laukamkit/capstone_project_GroupA.git

Cloning into 'capstone_project_GroupA'...
remote: Enumerating objects: 1194, done.
remote: Counting objects: 100% (310/310), done.
remote: Compressing objects: 100% (222/222), done.
remote: Total 1194 (delta 180), reused 182 (delta 86), pack-reused 884 (from 3)
Receiving objects: 100% (1194/1194), 331.47 MiB | 12.31 MiB/s, done.
Resolving deltas: 100% (586/586), done.
Updating files: 100% (99/99), done.


In [2]:
%cd capstone_project_GroupA
!git checkout colab

/content/capstone_project_GroupA
Branch 'colab' set up to track remote branch 'colab' from 'origin'.
Switched to a new branch 'colab'


In [3]:
%cd src

/content/capstone_project_GroupA/src


PatchTST BIG

In [4]:
from datetime import datetime
from ModelFiles.GroupAModels import TransformersModel
from ModelFiles.ModelConfigs import TransformersConfig, HORIZONS, SEEDS
from ModelFiles.ModelEnums import TransformerModelType
from ModelFiles.ModelPlots import *

USE_LOG_TARGET = True
CONTEXT_LENGTHS = [336, 512, 720] # two variants tested in PatchTST paper, 720 I added our own.
EVAL_STEP_SIZE = 48
NUM_EPOCHS = 100
PATIENCE = 10
DEBUG = False

for horizon in HORIZONS:
    for context_length in CONTEXT_LENGTHS:
        for seed in SEEDS:
            if seed == SEEDS[-1]:
                save_prediction_results = True
            else:
                save_prediction_results = False

            patchtst_config = TransformersConfig(
                task_id=f"patchtst_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
                model=TransformerModelType.PATCHTST,
                forecast_horizon=horizon,
                lookback_window=context_length,
                used_log_target=USE_LOG_TARGET,
                target_col= "LOG_TOTALDEMAND" if USE_LOG_TARGET else "TOTALDEMAND",
                feature_cols=['TEMPERATURE', 'TEMP_SQUARED', 'IS_WEEKEND', 'demand_1_year_ago'],
                scale=True,
                date_col='DATETIME',
                variate='MS',
                patch_len=16,
                stride=8,
                d_model=512,
                num_attention_heads=8,
                num_encoder_layers=3,
                dim_ff=2048,
                dropout=0.1,
                dropout_head_fc=0.1,
                use_gpu=True,
                time_encoding='timeF',
                training_epochs=NUM_EPOCHS,
                batch_size=32,
                learning_rate=0.0001,
                output_attention=False,
                lradj='TST',
                patience=PATIENCE,
                seed=seed,
                eval_step_size=EVAL_STEP_SIZE,
                save_test_results=save_prediction_results,
                debug=DEBUG,
                save_training_log=True,
            )
            patch_tst_model = TransformersModel(patchtst_config)
            patch_tst_model.train_model()
            patch_tst_model.evaluate_model(test_mode=1)
            print("=" * 200)
            print("\n")

Found NSW data path: /content/capstone_project_GroupA/data/NSW


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Set random seed to 31415
Use GPU: cuda:0
train 107012


KeyboardInterrupt: 